# Neuron Voxel Lab · Autoencoders, from a hand-written gradient to CelebA on your L4

Companion notebook to the **Neuron Voxel Lab** site (stations 1–10). Same toy faces, same network, same equations, so you can check every number the site shows.

**What the CelebA lesson in your screenshot is saying:** an autoencoder is trained to output *its own input*. The middle layer (the **bottleneck**) is much smaller than the image, so the network can't copy pixels. It has to *compress*: find a short description of a face that is still enough to redraw it. No labels are used for training. CelebA's 40 labels (`Smiling`, `Young`, `Bags_Under_Eyes`, …) are used only at the end, to find **directions** in the code space ("add eyebags").

| Part | What you do | Where in the books |
|---|---|---|
| 0 | Check the L4 | D2L ch. 6 (GPUs), ch. 13 |
| 1 | Toy-face autoencoder in **pure NumPy**: every gradient by hand, checked numerically, compared with PCA | Murphy §13.3, §20.1, §20.3.1 · D2L §5.3 |
| 2 | The same net in **PyTorch**; prove autograd gives *your* gradients | D2L §2.5 |
| 3 | Stream 50,000 CelebA faces, crop to 64×64, keep them on the GPU | Murphy §1.5 (data) |
| 4 | Train the **conv autoencoder** with bf16 on the L4 | D2L ch. 7, §8.5, §14.10 · Murphy §14.2 |
| 5 | Bottleneck experiment: 16 vs 64 vs 256 numbers | Murphy §20.3.1 |
| 6 | **Attribute vectors** (smile, eyebags, age) and face interpolation | Murphy §20.3.5, §20.5 |
| 7 | Your turn: denoising AE, VAE | Murphy §20.3.2, §20.3.5 |

Run it top to bottom. Part 1 and 2 take under a minute. Part 3 takes about 5–10 minutes the first time (the result is cached to your Drive). Part 4 takes a few minutes on an L4.

## Part 0 · Check the GPU
In Colab: **Runtime → Change runtime type → L4 GPU**. Then run:

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt, time, os, math
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Part 1 · The autoencoder in pure NumPy (station 7 of the site)

### 1.1 Toy faces
Same recipe as the site: a 12×12 grey image, a round face, two eyes, optional eyebags, and a mouth whose corners go up (smile) or down (frown). `smile` and `bags` are the labels; the autoencoder never sees them.

In [ ]:
def make_face(rng, smile=None, bags=None, ey=None, gap=None, shade=None):
    im = np.full((12, 12), 0.3)                                   # background
    yy, xx = np.mgrid[:12, :12]
    shade = 0.72 + 0.28 * rng.random() if shade is None else shade
    im[(yy - 5.5) ** 2 + (xx - 5.5) ** 2 <= 5.3 ** 2] = shade     # round face
    ey = 3 + rng.integers(2) if ey is None else ey                # eye row
    gap = 1 + rng.integers(3) if gap is None else gap             # eye spacing
    lx, rx = 5 - gap, 6 + gap
    for x in (lx, rx):
        im[ey:ey + 2, x] = 0.05                                   # eyes
    bags = rng.random() < 0.5 if bags is None else bags
    if bags:
        for x in (lx, rx):
            im[ey + 2, x] = 0.5                                   # eyebags
    s = rng.random() * 2 - 1 if smile is None else smile          # -1 frown .. +1 smile
    for x in range(3, 9):
        u = (x - 5.5) / 2.5
        im[8 - int(np.floor(s * u * u * 1.6 + 0.5)), x] = 0.05     # mouth
    return im, s > 0, bags

def make_faces(n, seed):
    rng = np.random.default_rng(seed)
    out = [make_face(rng) for _ in range(n)]
    X = np.stack([o[0].reshape(-1) for o in out])                 # (n, 144): flatten (station 6)
    return X, np.array([o[1] for o in out]), np.array([o[2] for o in out])

X, smile, bags = make_faces(1200, seed=101)
Xtest, smile_t, bags_t = make_faces(200, seed=202)
print(X.shape, "smiling:", smile.mean().round(2), "eyebags:", bags.mean().round(2))

def show_faces(rows, titles=None, size=1.1):
    rows = [np.asarray(r) for r in rows]
    fig, ax = plt.subplots(len(rows), len(rows[0]), figsize=(size * len(rows[0]), size * len(rows)), squeeze=False)
    for i, r in enumerate(rows):
        for j, im in enumerate(r):
            ax[i, j].imshow(im.reshape(12, 12), cmap="gray", vmin=0, vmax=1); ax[i, j].axis("off")
        if titles: ax[i, 0].set_title(titles[i], fontsize=8, loc="left")
    plt.tight_layout(); plt.show()

show_faces([X[:10]], ["ten toy faces"])

### 1.2 The network, every line by hand
`144 → 32 (tanh) → L (no squash: the bottleneck) → 32 (tanh) → 144 (sigmoid)`

* **Forward:** each layer is station 5's `h = φ(h_prev @ W + b)`. We keep every layer's output because backprop needs them.
* **Loss:** mean squared error between output and input, `mean((out − x)²)`.
* **Backward:** station 4, for a whole layer at a time. At each layer:
  `dz = d · φ'(h)` (through the squash), then `dW = h_prevᵀ @ dz`, `db = Σ dz`, and the blame for the layer below is `d = dz @ Wᵀ`.
* **Adam:** gradient descent (station 3) where each weight's step is its averaged slope divided by that slope's typical size (D2L §12.10).

In [ ]:
def act(z, kind):
    if kind == "tanh":    return np.tanh(z)
    if kind == "sigmoid": return 1 / (1 + np.exp(-z))
    return z                                                      # "none"

def dact(y, kind):   # slope of the squash, written in terms of its OUTPUT y
    if kind == "tanh":    return 1 - y ** 2
    if kind == "sigmoid": return y * (1 - y)
    return np.ones_like(y)

class NumpyAE:
    def __init__(self, L=4, linear=False, seed=0):
        rng = np.random.default_rng(seed)
        self.sizes = [144, 32, L, 32, 144]
        self.acts = ["none"] * 4 if linear else ["tanh", "none", "tanh", "sigmoid"]
        self.W = [rng.normal(0, 1 / np.sqrt(i), (i, o)) for i, o in zip(self.sizes, self.sizes[1:])]
        self.b = [np.zeros(o) for o in self.sizes[1:]]
        self.m = [np.zeros_like(p) for p in self.W + self.b]
        self.v = [np.zeros_like(p) for p in self.W + self.b]
        self.t = 0

    def forward(self, x):
        self.h = [x]
        for W, b, a in zip(self.W, self.b, self.acts):
            self.h.append(act(self.h[-1] @ W + b, a))
        return self.h[-1]

    def encode(self, x):
        return act(act(x @ self.W[0] + self.b[0], self.acts[0]) @ self.W[1] + self.b[1], self.acts[1])

    def decode(self, z):
        h = act(z @ self.W[2] + self.b[2], self.acts[2])
        return act(h @ self.W[3] + self.b[3], self.acts[3])

    def loss_and_grads(self, x):
        out = self.forward(x)
        loss = np.mean((out - x) ** 2)
        d = 2 * (out - x) / out.size                     # dLoss/dOut: derivative of a mean of squares
        gW, gb = [None] * 4, [None] * 4
        for k in reversed(range(4)):
            dz = d * dact(self.h[k + 1], self.acts[k])   # through the squash
            gW[k] = self.h[k].T @ dz                     # "times the other input" (station 4), for every weight at once
            gb[k] = dz.sum(0)                            # addition copies blame
            d = dz @ self.W[k].T                         # blame for the layer below
        return loss, gW, gb

    def adam_step(self, gW, gb, lr=4e-3, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        for i, (p, g) in enumerate(zip(self.W + self.b, gW + gb)):
            self.m[i] = b1 * self.m[i] + (1 - b1) * g            # running mean of the slope
            self.v[i] = b2 * self.v[i] + (1 - b2) * g ** 2       # running mean of slope²
            mh, vh = self.m[i] / (1 - b1 ** self.t), self.v[i] / (1 - b2 ** self.t)
            p -= lr * mh / (np.sqrt(vh) + eps)                   # in-place update

ae = NumpyAE(L=4, seed=1)
loss, gW, gb = ae.loss_and_grads(X[:32])
print("starting loss", round(loss, 4), "| gradient shapes:", [g.shape for g in gW])

### 1.3 Don't trust backprop: measure the slope
Nudge one weight up by h, then down by h, and measure how the loss moves (the site's station 4 "check" button). If the backward pass is right, the two numbers agree to many decimal places.

In [ ]:
xb = X[:32]
_, gW, _ = ae.loss_and_grads(xb)
h = 1e-6
for (k, i, j) in [(0, 10, 3), (1, 5, 2), (2, 1, 7), (3, 20, 100)]:
    old = ae.W[k][i, j]
    ae.W[k][i, j] = old + h; lp = np.mean((ae.forward(xb) - xb) ** 2)
    ae.W[k][i, j] = old - h; lm = np.mean((ae.forward(xb) - xb) ** 2)
    ae.W[k][i, j] = old
    print(f"layer {k} weight[{i},{j}]   backprop {gW[k][i, j]: .8f}   measured {(lp - lm) / (2 * h): .8f}")

### 1.4 Train it, and compare with PCA
PCA is the best any *straight-line* squeeze to L numbers can do (Murphy §20.1). Its error is the variance left in the directions it throws away: `(sum of all eigenvalues − sum of the top L) / 144`.

In [ ]:
def pca_error(X, L):
    Xc = X - X.mean(0)
    evals = np.linalg.eigvalsh(Xc.T @ Xc / len(X))[::-1]         # covariance eigenvalues, largest first
    return (evals.sum() - evals[:L].sum()) / X.shape[1]

def train_numpy(ae, X, steps=4000, batch=32, lr=4e-3, seed=0):
    rng = np.random.default_rng(seed); hist = []
    for s in range(steps):
        xb = X[rng.integers(0, len(X), batch)]
        loss, gW, gb = ae.loss_and_grads(xb)
        ae.adam_step(gW, gb, lr)
        if s % 100 == 0:
            hist.append(np.mean((ae.forward(X[:300]) - X[:300]) ** 2))
    return hist

L = 4
t0 = time.time()
ae = NumpyAE(L=L, seed=1);              h_nl = train_numpy(ae, X)
lin = NumpyAE(L=L, linear=True, seed=1); h_lin = train_numpy(lin, X, lr=2e-3)
print(f"trained both in {time.time() - t0:.1f}s on the CPU")
p = pca_error(X, L)
plt.figure(figsize=(6, 3))
plt.plot(np.arange(len(h_nl)) * 100, h_nl, label="autoencoder (tanh/sigmoid)")
plt.plot(np.arange(len(h_lin)) * 100, h_lin, label="linear autoencoder")
plt.axhline(p, ls="--", c="orange", label=f"PCA, L={L}")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("error per pixel"); plt.legend(); plt.show()
print(f"final  nonlinear {h_nl[-1]:.4f}   linear {h_lin[-1]:.4f}   PCA {p:.4f}")
print("The linear one should settle on the PCA line (Murphy 20.3.1). The nonlinear one can go below it.")
show_faces([Xtest[:10], ae.forward(Xtest[:10]), lin.forward(Xtest[:10])],
           ["test faces", "nonlinear AE, L=4", "linear AE (= PCA), L=4"])

### 1.5 The 2-number map (station 8)
With L = 2 every face gets a point on a plane. Colour the points by the labels the network never saw.

In [ ]:
ae2 = NumpyAE(L=2, seed=21); train_numpy(ae2, X, steps=5000)
Z = ae2.encode(X)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].scatter(Z[:, 0], Z[:, 1], c=smile, cmap="coolwarm", s=6); ax[0].set_title("colour = smiling")
ax[1].scatter(Z[:, 0], Z[:, 1], c=bags, cmap="viridis", s=6);   ax[1].set_title("colour = eyebags")
d_smile = Z[smile].mean(0) - Z[~smile].mean(0)
d_bags = Z[bags].mean(0) - Z[~bags].mean(0)
for a, d, c in [(ax[0], d_smile, "k"), (ax[1], d_bags, "r")]:
    m = Z.mean(0); a.arrow(m[0], m[1], d[0], d[1], width=0.02, color=c)
plt.show()
z0 = Z[np.where(~smile)[0][0]]
show_faces([ae2.decode(np.stack([z0 + t * d_smile for t in np.linspace(-1, 2, 8)]))], ["walking along the smile arrow"])

## Part 2 · Same network in PyTorch, same gradients
PyTorch's autograd records every operation and runs station 4's backward walk for you. To prove it does what your NumPy code does, copy the NumPy weights into a PyTorch model and compare gradients on the same batch. Note `nn.Linear` stores `W` transposed (out × in), hence `.T`.

In [ ]:
import torch.nn as nn, torch.nn.functional as F

def torch_toy_ae(L=4):
    return nn.Sequential(nn.Linear(144, 32), nn.Tanh(), nn.Linear(32, L),
                         nn.Linear(L, 32), nn.Tanh(), nn.Linear(32, 144), nn.Sigmoid())

fresh = NumpyAE(L=4, seed=7)
tm = torch_toy_ae(4).double()
lins = [m for m in tm if isinstance(m, nn.Linear)]
with torch.no_grad():
    for lin, W, b in zip(lins, fresh.W, fresh.b):
        lin.weight.copy_(torch.from_numpy(W.T)); lin.bias.copy_(torch.from_numpy(b))

xb = X[:32]
loss_np, gW, gb = fresh.loss_and_grads(xb)
xt = torch.from_numpy(xb)
loss_t = F.mse_loss(tm(xt), xt); loss_t.backward()
print("loss  numpy", loss_np, " torch", loss_t.item())
for k, lin in enumerate(lins):
    print(f"layer {k}: max |torch grad − numpy grad| = {np.abs(lin.weight.grad.numpy().T - gW[k]).max():.2e}")

In [ ]:
# Now train the toy net on the GPU, PyTorch style
dev = "cuda" if torch.cuda.is_available() else "cpu"
Xt = torch.tensor(X, dtype=torch.float32, device=dev)
toy = torch_toy_ae(4).to(dev)
opt = torch.optim.Adam(toy.parameters(), lr=4e-3)
for step in range(4000):
    xb = Xt[torch.randint(len(Xt), (32,), device=dev)]
    loss = F.mse_loss(toy(xb), xb)
    opt.zero_grad(); loss.backward(); opt.step()
print("toy torch AE error:", F.mse_loss(toy(Xt[:300]), Xt[:300]).item())

## Part 3 · CelebA: 50,000 real faces, 64 × 64, living on the GPU
The dataset is streamed from Hugging Face (`flwrlabs/celeba`: 162,770 training images, 178 × 218 px, with the 40 boolean attributes). Each image is centre-cropped to the face (148 × 148) and resized to 64 × 64. The result is cached to your Drive, so next time this cell loads in seconds.

50,000 × 3 × 64 × 64 bytes ≈ 614 MB: small enough to keep the **whole dataset on the L4** (24 GB). That way there's no data loader and the GPU never waits.

In [ ]:
!pip -q install datasets
from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm

N = 50_000
USE_DRIVE = True          # cache to Google Drive so you only download once
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = "/content/drive/MyDrive/neuron_voxel_lab/celeba64_50k.pt"
else:
    CACHE = "/content/celeba64_50k.pt"
os.makedirs(os.path.dirname(CACHE), exist_ok=True)

if os.path.exists(CACHE):
    pack = torch.load(CACHE)
    print("loaded cache", pack["images"].shape)
else:
    ds = load_dataset("flwrlabs/celeba", split="train", streaming=True)
    imgs = np.empty((N, 64, 64, 3), np.uint8)
    names, attrs = None, None
    for i, ex in enumerate(tqdm(ds.take(N), total=N)):
        im = ex["image"].convert("RGB").crop((15, 40, 163, 188)).resize((64, 64), Image.BILINEAR)
        imgs[i] = np.asarray(im)
        if names is None:
            names = [k for k in ex if k not in ("image", "celeb_id")]
            attrs = {k: np.zeros(N, bool) for k in names}
        for k in names:
            attrs[k][i] = ex[k] in (True, 1)
    pack = {"images": torch.from_numpy(imgs).permute(0, 3, 1, 2).contiguous(),     # (N, 3, 64, 64) uint8
            "attrs": {k: torch.from_numpy(v) for k, v in attrs.items()}}
    torch.save(pack, CACHE)
    print("saved", CACHE)

print(len(pack["attrs"]), "attributes, e.g.", list(pack["attrs"])[:6])

In [ ]:
def show_rgb(rows, titles=None, size=1.2):
    fig, ax = plt.subplots(len(rows), rows[0].shape[0], figsize=(size * rows[0].shape[0], size * len(rows)), squeeze=False)
    for i, r in enumerate(rows):
        r = r.detach().float().clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()
        for j in range(len(r)):
            ax[i, j].imshow(r[j]); ax[i, j].axis("off")
        if titles: ax[i, 0].set_title(titles[i], fontsize=8, loc="left")
    plt.tight_layout(); plt.show()

device = "cuda"
X_all = pack["images"].to(device)                       # uint8 on the GPU
A_all = {k: v.to(device) for k, v in pack["attrs"].items()}
n_val = 2_000
Xtr, Xval = X_all[:-n_val], X_all[-n_val:]
to_float = lambda xb: xb.float().div_(255).contiguous(memory_format=torch.channels_last)
show_rgb([to_float(Xtr[:10])], ["CelebA, cropped to 64×64"])
print("GPU memory used by the dataset:", round(X_all.numel() / 1e6), "MB")

## Part 4 · The conv autoencoder (station 10) trained on the L4
Encoder: four stride-2 convolutions, 64 → 32 → 16 → 8 → 4 pixels while channels grow 3 → 32 → 64 → 128 → 256, then a Linear layer to **128 numbers**. Decoder: the mirror image with transposed convolutions (D2L §14.10).

Speed settings used below, each one line:
* `torch.autocast(dtype=torch.bfloat16)`: run the multiply-adds in 16-bit "brain float" on the L4's tensor cores. Same range as float32, fewer digits; the loss is still computed in float32.
* `channels_last`: lay tensors out the way the tensor cores want them.
* `cudnn.benchmark`: let cuDNN time a few convolution algorithms once and keep the fastest.

In [ ]:
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def down(ci, co):
    return nn.Sequential(nn.Conv2d(ci, co, 4, 2, 1), nn.BatchNorm2d(co), nn.LeakyReLU(0.2))
def up(ci, co):
    return nn.Sequential(nn.ConvTranspose2d(ci, co, 4, 2, 1), nn.BatchNorm2d(co), nn.ReLU())

class ConvAE(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.enc = nn.Sequential(down(3, 32), down(32, 64), down(64, 128), down(128, 256),
                                 nn.Flatten(), nn.Linear(256 * 4 * 4, latent_dim))
        self.dec_fc = nn.Linear(latent_dim, 256 * 4 * 4)
        self.dec = nn.Sequential(up(256, 128), up(128, 64), up(64, 32),
                                 nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid())
    def encode(self, x): return self.enc(x)
    def decode(self, z): return self.dec(self.dec_fc(z).view(-1, 256, 4, 4))
    def forward(self, x): return self.decode(self.encode(x))

# Check every shape on paper first (station 10's formula), then in code:
m = ConvAE(128)
x = torch.zeros(2, 3, 64, 64); h = x
for layer in list(m.enc):
    h = layer(h); print(f"{layer.__class__.__name__:12s} -> {tuple(h.shape)}")
print("decoded:", tuple(m(x).shape), "| parameters:", f"{sum(p.numel() for p in m.parameters()):,}")

In [ ]:
@torch.no_grad()
def evaluate(model, Xv, bs=500):
    model.eval(); tot = 0.0
    for i in range(0, len(Xv), bs):
        xb = to_float(Xv[i:i + bs])
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out = model(xb)
        tot += F.mse_loss(out.float(), xb, reduction="sum").item()
    model.train()
    return tot / (len(Xv) * 3 * 64 * 64)

def train(latent_dim=128, epochs=12, bs=256, lr=2e-3, show=True):
    model = ConvAE(latent_dim).to(device).to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = len(Xtr) // bs
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * steps)
    hist = []
    for ep in range(epochs):
        t0 = time.time(); perm = torch.randperm(len(Xtr), device=device); run = 0.0
        for i in range(steps):
            xb = to_float(Xtr[perm[i * bs:(i + 1) * bs]])
            with torch.autocast("cuda", dtype=torch.bfloat16):
                out = model(xb)
            loss = F.mse_loss(out.float(), xb)                # station 7's loss, on 12,288 numbers
            opt.zero_grad(set_to_none=True)
            loss.backward()                                    # station 4, for ~2.4 million weights
            opt.step(); sched.step()                           # station 3
            run += loss.item() if i % 20 == 0 else 0
        torch.cuda.synchronize()
        v = evaluate(model, Xval); hist.append(v)
        print(f"latent {latent_dim:3d} | epoch {ep + 1:2d}/{epochs} | train {run / math.ceil(steps / 20):.4f} | val {v:.4f} | {time.time() - t0:.1f}s")
        if show and (ep == 0 or ep == epochs - 1):
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
                model.eval(); rec = model(to_float(Xval[:10])); model.train()
            show_rgb([to_float(Xval[:10]), rec], ["original", f"rebuilt from {latent_dim} numbers"])
    return model, hist

model, hist128 = train(latent_dim=128, epochs=12)

## Part 5 · How small can the bottleneck be?
Same network, three bottleneck sizes, four epochs each. Fewer numbers means blurrier faces, because the decoder has to guess more.

In [ ]:
results = {}
for Ld in (16, 64, 256):
    mdl, h = train(latent_dim=Ld, epochs=4, show=False)
    results[Ld] = (mdl, h)
plt.figure(figsize=(5, 3))
for Ld, (_, h) in results.items(): plt.plot(range(1, len(h) + 1), h, marker="o", label=f"{Ld} numbers")
plt.xlabel("epoch"); plt.ylabel("val error per pixel"); plt.legend(); plt.show()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    rows = [to_float(Xval[:8])]
    for Ld, (mdl, _) in results.items():
        mdl.eval(); rows.append(mdl(to_float(Xval[:8])))
show_rgb(rows, ["original"] + [f"{Ld} numbers" for Ld in results])
print("12,288 pixels → 16 numbers is 768× compression. Look at what survives: pose, skin tone, hair colour. Fine detail goes first.")

## Part 6 · Attribute vectors: "add eyebags"
This is where `age` and `eyebags` from your screenshot come in. Encode every face to its 128 numbers. Average the codes of faces **with** an attribute, subtract the average of faces **without** it, and you get an arrow (station 8). Add the arrow to any face's code and decode.

In [ ]:
@torch.no_grad()
def encode_all(model, Xs, bs=1000):
    model.eval(); zs = []
    for i in range(0, len(Xs), bs):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            zs.append(model.encode(to_float(Xs[i:i + bs])).float())
    return torch.cat(zs)

Z = encode_all(model, X_all)                       # (50000, 128)
print("codes:", tuple(Z.shape))

def direction(name):
    a = A_all[name]
    return Z[a].mean(0) - Z[~a].mean(0)

@torch.no_grad()
def walk(name, idx=(1, 5, 9, 14, 22, 31), ts=(-2, -1, 0, 1, 2)):
    d = direction(name); rows = []
    for t in ts:
        with torch.autocast("cuda", dtype=torch.bfloat16):
            rows.append(model.decode(Z[list(idx)] + t * d).float())
    show_rgb(rows, [f"{name}  t={t:+}" for t in ts])

for name in ("Smiling", "Bags_Under_Eyes", "Young", "Eyeglasses"):
    walk(name)

In [ ]:
# Interpolation: walk in a straight line from one face's code to another's
@torch.no_grad()
def interpolate(i, j, n=9):
    ts = torch.linspace(0, 1, n, device=device)[:, None]
    zs = (1 - ts) * Z[i] + ts * Z[j]
    with torch.autocast("cuda", dtype=torch.bfloat16):
        show_rgb([model.decode(zs).float()], [f"face {i} → face {j}"])
interpolate(3, 40); interpolate(100, 777)

torch.save(model.state_dict(), os.path.join(os.path.dirname(CACHE), "conv_ae_128.pt"))
print("model saved next to the data cache")

## Part 7 · Your turn
1. **Denoising autoencoder** (Murphy §20.3.2): feed `xb + 0.1 * torch.randn_like(xb)` in, but compare the output with the *clean* `xb`. One changed line in `train`. Do the attribute walks get smoother?
2. **Variational autoencoder** (Murphy §20.3.5): make the encoder output two vectors, `mu` and `logvar`; sample `z = mu + exp(logvar/2) * randn`; add `KL = -0.5 * mean(1 + logvar - mu² - exp(logvar))` to the loss. Then decode `torch.randn(16, 128)`: brand-new faces.
3. **Linear = PCA, at scale:** remove every BatchNorm and activation and compare the error with `torch.pca_lowrank` on the flattened images.
4. Ask the site's **"Ask about this"** box on station 7 or 8 to check your reasoning.